# Comparação de Algoritmos de Regressão na Base Diabetes

Atividade de Aplicação de Ciência de Dados.

## 1. Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

SEED = 42
sns.set_theme(style="whitegrid")

## 2. Base de dados

A base Diabetes é distribuída junto com o scikit-learn (`load_diabetes`). Ela reúne dados de 442 pacientes, com 10 variáveis preditoras (`age`, `sex`, `bmi`, `bp`, `s1` a `s6`). A variável alvo (`target`) é uma medida quantitativa da progressão da doença um ano após a avaliação inicial.

In [ ]:
dados = load_diabetes(as_frame=True)
df = dados.frame
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print('Valores ausentes por coluna:')
df.isna().sum()

### Distribuição da variável alvo

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(df["target"], bins=30, kde=True, ax=ax)
ax.set_xlabel("Progressão da doença (target)")
ax.set_ylabel("Frequência")
plt.show()

### Correlação entre as variáveis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
plt.show()

## 3. Preparação dos dados

Decisões desta etapa:

- A base não tem valores ausentes, então não há imputação.
- As variáveis preditoras já vêm centralizadas na média e escaladas pelo scikit-learn, então não é aplicada padronização adicional.
- Os dados são divididos em 80% para treino e 20% para teste, com `random_state` fixo. Essa mesma divisão será usada por todos os modelos, para que a comparação seja justa.

In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

print("Treino:", X_treino.shape, "| Teste:", X_teste.shape)

## 4. Modelos

Além do `DummyRegressor` (referência mínima, que sempre prevê a média do alvo no treino), foram escolhidos três algoritmos de famílias diferentes:

- **LinearRegression:** modelo linear simples, sem regularização.
- **Ridge:** modelo linear com regularização L2, que reduz o efeito da correlação entre variáveis (visível entre `s1`, `s2` e `s4`).
- **RandomForestRegressor:** conjunto de árvores de decisão, que captura relações não lineares e interações entre variáveis.

Todos usam os hiperparâmetros padrão do scikit-learn (exceto `n_estimators` e `random_state` da floresta), para que a comparação não seja influenciada por ajuste de hiperparâmetros feito em um modelo e não em outro.

In [ ]:
modelos = {
    "DummyRegressor": DummyRegressor(strategy="mean"),
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=200, random_state=SEED),
}

## 5. Avaliação e comparação

**Metodologia.** A avaliação segue duas etapas, iguais para os quatro modelos:

1. **Validação cruzada de 5 folds** apenas no conjunto de treino (com embaralhamento e `random_state` fixo), o que dá uma estimativa do desempenho e da variação entre folds.
2. **Avaliação final no conjunto de teste**, que não foi usado em nenhum ajuste, com o modelo treinado em todo o treino.

**Métricas.**
- **MAE:** erro absoluto médio, na mesma unidade do alvo.
- **RMSE:** raiz do erro quadrático médio, que penaliza mais os erros grandes.
- **R²:** proporção da variância do alvo explicada pelo modelo. Vale 0 para quem só prevê a média e pode ser negativo se o modelo for pior que isso.

Menor MAE e RMSE e maior R² indicam melhor desempenho.

### 5.1 Validação cruzada

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
metricas = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
}

linhas_cv = []
for nome, modelo in modelos.items():
    res = cross_validate(modelo, X_treino, y_treino, cv=kf, scoring=metricas)
    for fold in range(kf.get_n_splits()):
        linhas_cv.append({
            "Modelo": nome,
            "Fold": fold + 1,
            "MAE": -res["test_MAE"][fold],
            "RMSE": -res["test_RMSE"][fold],
            "R2": res["test_R2"][fold],
        })

cv_folds = pd.DataFrame(linhas_cv)
tabela_cv = cv_folds.groupby("Modelo")[["MAE", "RMSE", "R2"]].agg(["mean", "std"]).round(3)
tabela_cv = tabela_cv.loc[list(modelos)]
tabela_cv

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metrica in zip(axes, ["MAE", "RMSE", "R2"]):
    sns.boxplot(data=cv_folds, x="Modelo", y=metrica, order=list(modelos), ax=ax)
    ax.set_title(f"{metrica} por fold (validação cruzada)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### 5.2 Avaliação no conjunto de teste

In [ ]:
linhas_teste = []
previsoes = {}
for nome, modelo in modelos.items():
    modelo.fit(X_treino, y_treino)
    pred = modelo.predict(X_teste)
    previsoes[nome] = pred
    linhas_teste.append({
        "Modelo": nome,
        "MAE": mean_absolute_error(y_teste, pred),
        "RMSE": root_mean_squared_error(y_teste, pred),
        "R2": r2_score(y_teste, pred),
    })

tabela_teste = pd.DataFrame(linhas_teste).set_index("Modelo").round(3)
tabela_teste

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharex=True, sharey=True)
lim = [y_teste.min(), y_teste.max()]
for ax, (nome, pred) in zip(axes, previsoes.items()):
    ax.scatter(y_teste, pred, alpha=0.7)
    ax.plot(lim, lim, "r--", label="previsão perfeita")
    ax.set_title(nome)
    ax.set_xlabel("Valor real")
axes[0].set_ylabel("Valor previsto")
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
resumo = pd.concat(
    {"Validação cruzada (média)": tabela_cv.xs("mean", axis=1, level=1),
     "Teste": tabela_teste},
    axis=1,
)
resumo

## 6. Conclusões